# PROBLEMA 1 - INTER 2 Probabilidad y Estadística - iMAT-ICAI - 7 de marzo 2022

# Vale el 50% de la nota del examen
---

*NO CAMBIES EL NOMBRE AL NOTEBOOK, SIMPLEMENTE ESCRIBE AQUÍ TUS DATOS*
### Nombre: RELLENAR
### Apellidos: RELLENAR
---

### María y Carlos son dos alumnos extraordinariamente aplicados de primero del grado de IMAT. Ambos desean sentarse en primera fila, pero los asientos disponibles son limitados. Por ello cada día de clase, Carlos y María lanzan una moneda al aire para decidir quién se sienta en primera fila. Si sale cara, Carlos se sienta en primera fila, mientras que si sale cruz es María quien se sienta en primera fila.

### Carlos ha notado que María se sienta en primera fila un mayor número de veces que él. Por lo que Carlos sospecha que la moneda que usa María podría estar trucada. Para despejar sus dudas, durante 200 días Carlos anota en un Excel un 1 si sale cruz y un 0 si sale cara.

Responde las preguntas utilizando instrucciones print() para mostrar claramente tu respuesta en la ejecución de la celda, explicando cómo lo resuelves. Utiliza comentarios en tu código para facilitar su lectura.

### Importamos todos los paquetes aquí por comodidad

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy import optimize as opt  # para estimación por máxima verosimilitud

In [ ]:
df_orig = pd.read_csv('muestra_PROB1_INTER2024.csv')
print('Tamaño de df_orig con los datos cargados:', df_orig.shape)
# datos de la muestra para trabajar
x = df_orig.data.tolist()    #   <-------------------------------- x (valores en la muestra)
n = len(x)                   #   <-------------------------------- n (tamaño de la muestra)
 
# los datos de las semillas
print('Valores obtenidos 1: Cruz  0:Cara')
plt.figure(1, figsize=(15, 4))
plt.plot(x)
plt.xlabel('Día')
plt.ylabel('Cara/Cruz')
plt.grid()
plt.show()

# observamos su distribución
print('Distribución de caras y cruces:')
plt.figure(1, figsize=(15, 4))
plt.hist(x, bins = 40);plt.xlabel('x');plt.title(f'Muestra (n={n})')
plt.show()

### Cada observación de la muestra puede ser modelada por una distribución de Bernoulli de parámetro $p\in(0,1)$. Es decir $$X = \left\{\begin{array}
11 & \text{ con probabilidad } p\\
 0 & \text{ con probabilidad } 1-p.
 \end{array}\right.$$

### El objetivo de Carlos es estudiar cuál puede ser el valor desconocido del parámetro $p$. 

### A Carlos le gustaría realizar una primera suposición sobre el valor de $p$. Para ello utiliza lo que ha aprendido en la clase de estadística sobre el estimador de máxima verosimilitud. 

### Carlos define la función verosimilitud en Python. La cual dado un valor del parámetro $p$, esta devuelve el valor de la verosimilitud.

### La siguiente celda de python utiliza la función creada por Carlos para dibujar la función de verosimilitud en funcion de distintos valores de $p$.

In [ ]:
def verosimilitud(parametro_p):
    # sacamos los parámetros
    param = parametro_p
    
    #calculo el logarítmo de la verosimilitud
    logLikelihood = np.sum(stats.bernoulli.logpmf(x,param))  # <--- logpdf es el log de la pmf.
    
    #devuelvo la verosimilitud
    return np.exp(logLikelihood) 

# llamada a la función para mostrar cómo varía con el valor de p
p_parametros = np.linspace(0.001,0.999,1000)
likelihood = []
for p in p_parametros:
    new_value = verosimilitud(p)
    likelihood.append(new_value)

#Pinta la función de verosimilitud con respecto a varios valores del parámetro p
fig = plt.figure(1, figsize=(10, 10))
plt.plot(p_parametros, likelihood, color="orange", label = 'pdf(x)')
plt.xlabel('p')
plt.ylabel('L')
plt.title("Función de Verosimilitud")
plt.legend()
plt.grid()

### a) Basándote en la gráfica anterior, estima el parámetro $p$.
### Usando la función verosimilitud definida en la celda anterior calcula el valor de la verosimilitud cuando $p=0.7$ y para cuando $p=0.5$. (1 punto)

In [ ]:
# Respuesta

print("Como deseamos maximizar la función de verosimilitud, del gráfico obtenemos que una posible estimación razonable del parámetro \
p estaría alrededor del valor 0.8")
print("\n")
print("El valor de la verosimilitud cuando p=0.7 es :", verosimilitud(0.7))
print("El valor de la verosimilitud cuando p=0.5 es :", verosimilitud(0.5))

### Carlos desea construir un intervalo de confianza para el parámetro $p$. En un libro de estadística que ha encontrado en la biblioteca, Carlos obtiene el siguiente intervalo confianza con un nivel de confianza $\gamma$ para el parámetro poblacional $p$:$$\left[\bar{X} +  z_{\alpha/2} \left(\frac{\bar{X}(1-\bar{X}) }{n}\right)^{1/2}, \bar{X} +  z_{1-\alpha/2} \left(\frac{\bar{X}(1-\bar{X}) }{n}\right)^{1/2} \right]_{\gamma},$$ donde $\bar{X}$ es la media muestral y $z_{\alpha/2}$ es el valor crítico determinado como el percentil $\alpha/2$ de una normal estándar.

### Para construir dicho intervalo se ha usado el siguiente pivote estadístico:
$$\frac{\bar{X} -p}{\left(\frac{\bar{X}(1-\bar{X}) }{n}\right)^{1/2}} \sim N(0,1).$$

### b) Usa la fórmula que ha encontrado Carlos en la biblioteca y calcula el intervalo de confianza del parámetro $p$ con un nivel de confianza del $95\%$. Observando el resultado obtenido, ¿tiene sentido pensar que la moneda podría estar trucada? (2 puntos)

In [ ]:
# Respuesta 

# media muestral
x_media = np.mean(x)

se = np.sqrt(x_media*(1-x_media)/n)

Xmedia = stats.norm(loc=x_media, scale = se)  

#Para calcular un IC de la media de X usamos la distribución teórica de la media muestral
gama = 0.95
alfa = 1 - gama
lim_inf = Xmedia.ppf(alfa/2)
lim_sup = Xmedia.ppf(1-alfa/2)

print(f'El intervalo al {100*gama}% de la media de X es: [{lim_inf}, {lim_sup}]')
print("Podemos observar que el valor 0.5, no se encuentra en el intervalo de confianza obtenido.\
 Tiene sentido pensar que la moneda podría estar trucada")

In [ ]:
# Otra posible respuesta

Z = stats.norm(loc = 0, scale = 1)

lim_inf = x_media + se*Z.ppf(alfa/2)
lim_sup = x_media + se*Z.ppf(1-alfa/2)

print(f'El intervalo al {100*gama}% de la media de X es: [{lim_inf}, {lim_sup}]')

### c) Representa en la misma gráfica, la función de densidad de la distribución del estadistico utilizado para calcular el intervalo de confianza del apartado anterior, así como el intervalo de confianza construido. Indica claramente el área asociada al nivel de confianza utilizado. (1 punto)

In [ ]:
# Respuesta

xs = np.linspace(Xmedia.ppf(0.00001), Xmedia.ppf(1-0.00001), 1000)
xIC = np.linspace(lim_inf, lim_sup, 1000)

plt.figure(1, figsize=(15, 6))
plt.plot(xs, Xmedia.pdf(xs), linewidth = 4, label=f'Distribución Media muestral (según una t)')
plt.fill_between( xIC, Xmedia.pdf(xIC), color="green", alpha=0.3, label=f'Probabilidad asociada ($\gamma=${gama})')
plt.vlines(x_media, 0, np.max(Xmedia.pdf(xs)), color='black', label=f'Estimación puntual media {x_media}')
plt.errorbar(x_media, -0.01, xerr = np.array([[x_media - lim_inf], [lim_sup - x_media]]), fmt = 'o', label = f'IC [{lim_inf}, {lim_sup}] al {gama*100}%')
plt.title('Distribución del estimador media muestral de X e IC')
plt.tight_layout()
plt.legend()
plt.show()

print("Se confirma que el IC no contiene el valor 0.5, la moneda parece estar trucada.")

### d) Carlos desea confirmar que los resultados obtenidos usando la fórmula del libro son correctos. Para ello construye un intervalo de confianza para la media de $X$, con un nivel de confianza del $95\%$, usando el método de Bootstrap, para compararlo con el intervalo de confianza estimado en el apartado b). ¿Puede estar seguro Carlos de que la fórmula encontrada en la biblioteca es correcta? (2 puntos)

In [ ]:
# Respuesta

# params para llamar a bootstrap (método BCa, el que se suele usar normalmente)
muestra_orig = (x,)  # tiene que ser una secuencia
R = 9999
estadistico = np.mean
nivel_confianza = 0.95

res_BCa = stats.bootstrap(muestra_orig, estadistico, confidence_level = nivel_confianza,  n_resamples = R, method = 'BCA', random_state = 999)

# devuelve como resultado el IC y el SE del estadístico
lim_inf_BCa, lim_sup_BCa = res_BCa.confidence_interval

print(f'IC {100*nivel_confianza}% estimado según método BCa para la media: [{lim_inf_BCa}, {lim_sup_BCa}]')
print(f'El intervalo usando la fórmula del libro al {100*gama}% de la media de X es: [{lim_inf}, {lim_sup}]')
print("Podemos observar que ambos intervalos de confianza son similares, por lo que podemos confiar en la fórmula de la biblioteca.")

### e) Carlos quiere demostrar que la moneda usada por María obtiene cruz con mayor probabilidad que cara. Para ello quiere plantear un contraste de hipótesis. 

### En el libro que encontró en la biblioteca, se dice lo siguiente acerca del contraste de hipótesis unilateral para el parámetro poblacional $p$,
$$\begin{array}
H H_0 : & p = p_0\\
 H_1 : & p > p_0.
 \end{array}
$$ 

### Para este contraste de hipótesis rechazamos $H_0$ si:
$$t_0 > z_{1-\alpha},$$ 
### donde $t_0$ es el valor del estadístico de contraste en la muestra. El estadístico de contraste $T_0$ se define como

$$T_0 = \frac{\bar{X} - p_0}{\left(\frac{\bar{X}(1-\bar{X})}{n}\right)^{1/2}}.$$ 

### Además $T_0 \sim N(0,1)$ cuando $H_0$ es cierta.

### Ayuda a Carlos a construir el contraste de hipótesis, indicando las hipótesis H0 y H1. Resuelve el contraste de forma detallada a partir de la muestra disponible, con un nivel de significación del 1%.

### Calcula el valor del estadístico de contraste en la muestra, los valores críticos y el p-valor asociado. Razona utilizando por separado los valores críticos y el p-valor para tener más seguridad en lo que haces. ¿Con los resultados obtenidos podemos concluir que la probabilidad de obtener una cruz es mayor que la probabilidad de obtener una cara? (4 puntos)


In [ ]:
# Respuesta

print("Ya que queremos verificar que la probabilidad de sacar cruz sea mayor que la de sacar cara, queremos comprobar si p>0.5.")
print("Por lo que planteamos las siguientes hipótesis")
print("  H0: p = 0.5")
print("  H1: p > 0.5")
print("\n")

p0 = 0.5
x_media = np.mean(x)
se = np.sqrt(x_media*(1-x_media)/n)

t0 = (x_media - p0)/se
print("El valor del estadístico de contraste en la muestra es t0 =", t0)

T0 = stats.norm(loc = 0, scale = 1)
alfa = 0.01
t_crit_de = T0.ppf(1-alfa)
print("El valor crítico es t_crit_de = ", t_crit_de)
if(t0>t_crit_de):
    print("Rechazamos H_0 con un nivel de significación alfa =", alfa)
    print("Por lo que podemos afirmar que la probabilidad de sacar cruz es mayor que sacar cara.")
    print(f"Debido a que t0={t0} es mayor que t_crit_de = {t_crit_de}")
else:
    print("No rechazamos H_0 con un nivel de significación alfa =", alfa)
    print("Por lo que no podemos afirmar que la probabilidad de sacar cruz es mayor que sacar cara.")
    print(f"Debido a que t0={t0} es menor que t_crit_de = {t_crit_de}")
        
print("\n")
p_valor = 1-T0.cdf(t0)
print("El p-valor es igual a ", p_valor)
if(p_valor<alfa):
    print("Rechazamos H_0 con un nivel de significación alfa =", alfa)
    print("Por lo que podemos afirmar que la probabilidad de sacar cruz es mayor que sacar cara.")
    print(f"Debido a que p-valor={p_valor} es menor que alfa = {alfa}")
else:
    print("No rechazamos H_0 con un nivel de significación alfa =", alfa)
    print("Por lo que no podemos afirmar que la probabilidad de sacar cruz es mayor que sacar cara.")
    print(f"Debido a que p-valor={p_valor} es mayor que alfa = {alfa}")